# RAG Evaluation

Ablation study across 4 configs using Mistral-7B loaded on T4.

- change runtime to GPU first
- upload zip in cell below
- fill in api keys

In [ ]:
!pip install -q pinecone sentence-transformers transformers \
    bitsandbytes accelerate rank_bm25 nltk scikit-learn

In [ ]:
!pip install accelerate bitsandbytes --quiet

### upload files

In [ ]:
# upload zip
from google.colab import files
import zipfile, os

uploaded = files.upload()          # pick your NLP_RAG.zip
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

import glob
candidates = [p for p in glob.glob('/content/NLP_RAG*') if os.path.isdir(p)]
PROJECT_ROOT = candidates[0] if candidates else '/content/NLP_RAG'
print('Project root:', PROJECT_ROOT)

# or use drive instead:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/NLP_RAG'   # adjust path if needed

Saving NLP_RAG.zip to NLP_RAG (2).zip
Project root: /content/NLP_RAG


### api keys

In [ ]:
import os, sys

os.environ['PINECONE_API_KEY'] = 'pcsk_4K8982_3Eitg2NE3RQWeFhBitPt17dGrFGipBFFB53dPWzHUCh47Ho39rLoDBg9UANqgZ3'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'  # add your HF token here   # only needed to download the model once

PINECONE_API_KEY = os.environ['PINECONE_API_KEY']
HF_TOKEN = os.environ['HF_TOKEN']
INDEX_NAME       = 'rag-qa-nlp'

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())

Working directory: /content/NLP_RAG


In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB total')

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
generator = pipeline('text-generation', model=model, tokenizer=tokenizer)
print('Model loaded.')

GPU available: True
GPU: Tesla T4
Free VRAM: 15.5 GB / 15.6 GB total


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model loaded.


In [ ]:
def _local_generate(prompt, hf_token=None, max_new_tokens=300, temperature=0.1, retries=3):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return out[0]['generated_text'].strip()

import src.generation as _gen_module
import src.evaluation as _eval_module
_gen_module.call_hf_api = _local_generate
_eval_module.call_hf_api = _local_generate

print(_local_generate('<s>[INST] Say hello in one word. [/INST]'))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Sanity check: Hello!


In [ ]:
import json, time
from sentence_transformers import SentenceTransformer
from src.retrieval import HybridRetriever
from src.generation import generate_answer
from src.evaluation import LLMJudge

CONFIGS = [
    {'name': 'fixed_semantic', 'namespace': 'fixed', 'bm25': 'app/bm25_fixed.pkl', 'mode': 'semantic'},
    {'name': 'fixed_hybrid', 'namespace': 'fixed', 'bm25': 'app/bm25_fixed.pkl', 'mode': 'hybrid_rerank'},
    {'name': 'recursive_semantic', 'namespace': 'recursive', 'bm25': 'app/bm25_recursive.pkl', 'mode': 'semantic'},
    {'name': 'recursive_hybrid', 'namespace': 'recursive', 'bm25': 'app/bm25_recursive.pkl', 'mode': 'hybrid_rerank'},
]

def load_queries(path):
    queries = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            queries.append(json.loads(line))
    return queries

def run_config(config, queries, embedder):
    print(f"\n{'='*60}\nConfig: {config['name']}\n{'='*60}")
    retriever = HybridRetriever(
        pinecone_api_key=PINECONE_API_KEY,
        index_name=INDEX_NAME,
        namespace=config['namespace'],
        bm25_pkl_path=config['bm25'],
    )
    judge = LLMJudge(hf_token=HF_TOKEN, embedder=embedder)
    results = []
    for q in queries:
        print(f"\nQ{q['query_id']}: {q['question'][:70]}...")
        chunks = retriever.retrieve(q['question'], mode=config['mode'])
        answer, context = generate_answer(q['question'], chunks, HF_TOKEN)
        print(f'  Answer: {answer[:80]}...')
        time.sleep(0.5)
        eval_result = judge.evaluate_single(q['question'], answer, context)
        eval_result['query_id'] = q['query_id']
        eval_result['config'] = config['name']
        results.append(eval_result)
        print(f"  Faithfulness: {eval_result['faithfulness']:.3f} | Relevancy: {eval_result['relevancy']:.3f}")
    out_path = f"eval/results_{config['name']}.json"
    os.makedirs('eval', exist_ok=True)
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f'Saved: {out_path}')
    return results

print('Loading shared embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
queries = load_queries('eval/test_queries.jsonl')
print(f'Loaded {len(queries)} test queries')

all_results = {}
for config in CONFIGS:
    all_results[config['name']] = run_config(config, queries, embedder)

Loading shared embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

  Faithfulness: 1.000 | Relevancy: 0.036

Qq14: How does information retrieval differ from database querying?...
  Answer: Answer: Information retrieval differs from database querying in that the results...


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `m

  Faithfulness: 0.500 | Relevancy: 0.500

Qq15: What role did John McCarthy play in the founding of AI as a field?...
  Answer: Answer: John McCarthy played a significant role in the founding of AI as a field...


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `m

  Faithfulness: 0.625 | Relevancy: 0.007
Saved: eval/results_recursive_hybrid.json


### results

In [ ]:
import pandas as pd

rows = []
for config_name, results in all_results.items():
    avg_f = sum(r['faithfulness'] for r in results) / len(results)
    avg_r = sum(r['relevancy'] for r in results) / len(results)
    rows.append({'Config': config_name, 'Faithfulness': round(avg_f, 3), 'Relevancy': round(avg_r, 3), 'Combined': round((avg_f + avg_r) / 2, 3)})

df = pd.DataFrame(rows).sort_values('Combined', ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

            Config  Faithfulness  Relevancy  Combined
recursive_semantic         0.959      0.398     0.679
  recursive_hybrid         0.942      0.282     0.612
      fixed_hybrid         0.756      0.463     0.609
    fixed_semantic         0.819      0.393     0.606


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/eval_results', 'zip', 'eval')
files.download('/content/eval_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>